# Step 3 — Subdomains and bulk conversion

**Learning objective:** Convert the Gmsh mesh to OGS VTU files, then inspect cell orientations and subdomain labels.

**Run from top to bottom in VS Code.** Each notebook executes one workshop step and shows its output. Select the working OGS Postprocessing Python kernel.

**Prerequisites:** run earlier numbered steps. `GMSH2OGS` and `ogs` must be on PATH when used; Gmsh is imported from the selected Python kernel in Step 2. **Every image below represents a distinct quantity.** Green PASS messages verify the stated condition only.

In [ ]:
from pathlib import Path
import sys, subprocess
import numpy as np
import matplotlib.pyplot as plt
import meshio
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents, Path.cwd().parent) if (p / "models" / "excavation.prj").exists()), None)
assert ROOT is not None, "Open the extracted project directory in VS Code before running this notebook"
print('Training root:', ROOT)
print('Python:', sys.executable)
def run(*command):
    command = [str(c) for c in command]
    print('RUN:', ' '.join(command), flush=True)
    subprocess.run(command, cwd=ROOT, check=True)


**Explain:** `GMSH2OGS -e -b` exports bulk/subdomain and tagged boundary meshes. The orientation repair is necessary for the validated training mesh. **Run:** conversion and bulk orientation.

In [ ]:
run('GMSH2OGS','-i','mesh/excavation.msh','-o','mesh/excavation.vtu','-e','-b')
run(sys.executable,'scripts/orient_bulk_mesh.py')

### Terminal alternative — Step 3: GMSH2OGS bulk and tagged submeshes

Continue from **Step 2's terminal-generated** `terminal_mesh_output/excavation.msh` while staying in the course root. Copy these commands into the VS Code terminal:

```bash
GMSH2OGS -i terminal_mesh_output/excavation.msh -o terminal_mesh_output/excavation.vtu -e -b
python3 scripts/orient_bulk_mesh.py terminal_mesh_output
ls -lh terminal_mesh_output/*.vtu
```

`-e -b` exports bulk and numbered boundary/submesh files. The orientation fix updates **only** the terminal bulk VTU; it does not modify `mesh/excavation.vtu` used by the normal notebook sequence. **One soil region** is present (MaterialID 0); 1D boundary files are not extra materials.


**Inspect — bulk mesh versus actual material regions:** Each triangle carries a `MaterialIDs` label. This example has **one soil region: MaterialID = 0 for all elements**. A single colour is therefore the *correct subdomain plot*, not missing material information. The fine gray lines show the triangular discretization only. `Physical Curve` tags in Gmsh describe **1D boundaries**, not 2D material regions.

**Check:** All triangles must be labeled 0; the excluded wall has no elements. Different soil materials would require multiple `Physical Surface` regions and corresponding OGS media.

In [ ]:
m = meshio.read(ROOT / 'mesh' / 'excavation.vtu')
tri = np.vstack([b.data for b in m.cells if b.type == 'triangle'])
tags = m.cell_data.get('MaterialIDs')
assert tags is not None, 'MaterialIDs are absent — inspect the GMSH2OGS conversion'
material_ids = np.concatenate([np.asarray(v).reshape(-1) for v, b in zip(tags, m.cells) if b.type == 'triangle']).astype(int)
ids, counts = np.unique(material_ids, return_counts=True)
assert len(material_ids) == len(tri)
assert np.array_equal(ids, [0]), f'Expected one MaterialID 0, obtained {ids}'
print('Material regions:', {int(i): int(n) for i, n in zip(ids, counts)})
fig, axs = plt.subplots(1, 2, figsize=(14, 4.6), layout='constrained', sharex=True, sharey=True)
axs[0].triplot(m.points[:,0], m.points[:,1], tri, color='#64748b', lw=0.35)
axs[0].set_title(f'(A) Discretization: {len(m.points)} nodes, {len(tri)} triangles')
from matplotlib.colors import ListedColormap
pc = axs[1].tripcolor(m.points[:,0], m.points[:,1], tri, facecolors=material_ids,
                      cmap=ListedColormap(['#94c5df']), vmin=-0.5, vmax=0.5, edgecolors='#526273', linewidth=0.12)
axs[1].set_title(f'(B) Material region: ID 0 ({counts[0]} elements)')
for ax in axs:
    ax.set(xlim=(0,65),ylim=(0,30),xlabel='x [m]',ylabel='y [m]')
    ax.set_aspect('equal')
axs[0].text(0.02,0.03,'Triangles = numerical mesh',transform=axs[0].transAxes,bbox={'facecolor':'white','alpha':0.85,'edgecolor':'none'})
axs[1].text(0.02,0.03,'One colour = one 2D material region',transform=axs[1].transAxes,bbox={'facecolor':'white','alpha':0.85,'edgecolor':'none'})
plt.show()
print(f'PASS: all {len(tri)} triangles belong to MaterialID 0; wall slit has no cells')


**Question:** Why is the plot uniform in colour? **Answer:** Because all 1,236 triangles belong to the same material region. Boundary submeshes are introduced separately in Step 4.